# Charlene · Interaction Hierarchy & Funnel and Feedback Semantics

Analysis of the combined MIND-small train and validation logs. Only article exposure and binary clicks are observed; detail reads, carts, and purchases are not recorded.

## Setup and exposure preparation


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

sns.set_theme(style="whitegrid")

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "data_preparation"
    else Path.cwd()
)

TRAIN_DIR = PROJECT_ROOT / "data" / "raw" / "train"
VALID_DIR = PROJECT_ROOT / "data" / "raw" / "valid"

BEHAVIOR_COLUMNS = [
    "impression_id",
    "user_id",
    "time",
    "history",
    "impressions"
]



In [ ]:
train_behaviors = pd.read_csv(
    TRAIN_DIR / "behaviors.tsv",
    sep="\t",
    header=None,
    names=BEHAVIOR_COLUMNS
)

valid_behaviors = pd.read_csv(
    VALID_DIR / "behaviors.tsv",
    sep="\t",
    header=None,
    names=BEHAVIOR_COLUMNS
)

# Retain the original source
train_behaviors["source_split"] = "train"
valid_behaviors["source_split"] = "validation"

behaviors = pd.concat(
    [train_behaviors, valid_behaviors],
    ignore_index=True
)

behaviors["time"] = pd.to_datetime(
    behaviors["time"],
    format="%m/%d/%Y %I:%M:%S %p"
)

# Impression IDs may restart in each source file
behaviors["global_impression_id"] = (
    behaviors["source_split"]
    + "_"
    + behaviors["impression_id"].astype(str)
)

behaviors = behaviors.sort_values("time").reset_index(drop=True)

print("Training impressions:", f"{len(train_behaviors):,}")
print("Validation impressions:", f"{len(valid_behaviors):,}")
print("Combined impressions:", f"{len(behaviors):,}")
print("Unique users:", f"{behaviors['user_id'].nunique():,}")
print(
    "Observation period:",
    behaviors["time"].min(),
    "to",
    behaviors["time"].max()
)

display(behaviors.head())

### Expanding Impression Slates

The raw `impressions` column stores multiple exposed articles in one string.
Each item follows the format `news_id-label`, where label 1 represents a click
and label 0 represents an exposed but unclicked article.

We expand this column so that each row represents one article exposure.

In [ ]:
# Keep the relevant impression-level information
exposures = behaviors[
    ["global_impression_id", "impression_id", "user_id", "time", "impressions"]
].copy()

# Split each slate into a list of article-label pairs
exposures["candidate"] = exposures["impressions"].str.split()

# Create one row for every article exposure
exposures = exposures.explode("candidate", ignore_index=True)

# Separate the article ID and click label
exposures[["news_id", "label"]] = exposures["candidate"].str.rsplit(
    "-",
    n=1,
    expand=True
)

# Convert the label from text to an integer
exposures["label"] = exposures["label"].astype(int)

# Record each article's position within its impression slate
exposures["position"] = (
    exposures.groupby("global_impression_id").cumcount() + 1
)

# Keep only the useful columns
exposures = exposures[
    [
        "global_impression_id",
        "impression_id",
        "user_id",
        "time",
        "news_id",
        "position",
        "label"
    ]
]

display(exposures.head(10))
print("Exposure-level shape:", exposures.shape)

In [ ]:
print("Unique labels:", sorted(exposures["label"].unique()))
print("Missing values:")
display(exposures.isna().sum())

print(
    "Unique impressions before expansion:",
    behaviors["global_impression_id"].nunique()
)

print(
    "Unique impressions after expansion:",
    exposures["global_impression_id"].nunique()
)

print(
    "Duplicate impression-article pairs:",
    exposures.duplicated(["global_impression_id", "news_id"]).sum()
)

In [ ]:
impression_summary = (
    exposures.groupby("global_impression_id")
    .agg(
        user_id=("user_id", "first"),
        time=("time", "first"),
        slate_length=("news_id", "size"),
        click_count=("label", "sum")
    )
    .reset_index()
)

impression_summary["non_click_count"] = (
    impression_summary["slate_length"]
    - impression_summary["click_count"]
)

impression_summary["impression_ctr"] = (
    impression_summary["click_count"]
    / impression_summary["slate_length"]
)

display(impression_summary.head())

In [ ]:
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

## Interaction Hierarchy and Funnel

MIND records article exposures and binary click outcomes but does not provide
downstream interaction stages such as reading duration, sharing or subscription.
The observable item-level funnel is therefore limited to exposure and click.

In [ ]:
total_impressions = len(impression_summary)
total_exposures = len(exposures)
total_clicks = exposures["label"].sum()
total_non_clicks = total_exposures - total_clicks

overall_ctr = total_clicks / total_exposures

zero_click_impressions = (
    impression_summary["click_count"].eq(0).sum()
)

one_click_impressions = (
    impression_summary["click_count"].eq(1).sum()
)

multiple_click_impressions = (
    impression_summary["click_count"].gt(1).sum()
)

interaction_statistics = pd.Series({
    "Total impressions": total_impressions,
    "Total article exposures": total_exposures,
    "Total clicked exposures": total_clicks,
    "Total unclicked exposures": total_non_clicks,
    "Overall item-level CTR": overall_ctr,
    "Mean slate length": impression_summary["slate_length"].mean(),
    "Median slate length": impression_summary["slate_length"].median(),
    "Minimum slate length": impression_summary["slate_length"].min(),
    "Maximum slate length": impression_summary["slate_length"].max(),
    "Zero-click impressions": zero_click_impressions,
    "One-click impressions": one_click_impressions,
    "Multiple-click impressions": multiple_click_impressions
})

display(interaction_statistics)

In [ ]:
impression_click_rate = (
    impression_summary["click_count"].gt(0).mean()
)

print(f"Overall item-level CTR: {overall_ctr:.4f}")
print(f"Impression-level click rate: {impression_click_rate:.4f}")

In [ ]:
funnel_summary = pd.DataFrame({
    "Stage": [
        "Article exposures",
        "Clicked exposures"
    ],
    "Count": [
        total_exposures,
        total_clicks
    ],
    "Percentage of exposures": [
        100,
        overall_ctr * 100
    ]
})

display(funnel_summary)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

bars = ax.barh(
    funnel_summary["Stage"],
    funnel_summary["Count"],
    color=["#4C78A8", "#59A14F"]
)

ax.set_xscale("log")
ax.set_title("Observed Interaction Funnel in MINDsmall")
ax.set_xlabel("Number of Recorded Events (Log Scale)")
ax.set_ylabel("Interaction Stage")

ax.bar_label(
    bars,
    labels=[
        f"{count:,}"
        for count in funnel_summary["Count"]
    ],
    padding=5
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "interaction_funnel.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

The funnel counts article exposures and clicked exposures. The item-level conversion rate is clicks divided by exposures; the impression-level click rate measures slates with at least one click. These have different denominators. The observed slate click rate reflects the supplied dataset and should not be interpreted as platform-wide engagement. No post-click conversion can be measured from these logs.


## Feedback Semantics

MIND distinguishes between clicked articles, exposed-but-unclicked articles and
articles for which no user-specific exposure was recorded. These feedback states
should not be interpreted as equally strong preference signals.

| State | Recorded label | Interpretation |
|---|---|---|
| Exposed and clicked | 1 | Observed positive implicit feedback |
| Exposed but unclicked | 0 | Observed non-click; a potentially noisy negative training signal |
| No recorded exposure for this user–article pair | None | Unobserved feedback, not a zero label or evidence of dislike |

A non-click is explicitly recorded in the exposure log, but it is not an explicit dislike rating. An article absent from an impression must not automatically be assigned label 0. No recorded exposure does not prove the user never saw the article outside these logs.


In [ ]:
feedback_summary = pd.DataFrame({
    "Feedback type": [
        "Clicked",
        "Exposed but unclicked"
    ],
    "Label": [1, 0],
    "Count": [
        total_clicks,
        total_non_clicks
    ],
    "Percentage": [
        total_clicks / total_exposures * 100,
        total_non_clicks / total_exposures * 100
    ]
})

display(feedback_summary)

In [ ]:
user_article_feedback = (
    exposures.groupby(["user_id", "news_id"])
    .agg(
        exposure_count=("label", "size"),
        click_count=("label", "sum"),
        different_labels=("label", "nunique")
    )
    .reset_index()
)

repeated_pairs = user_article_feedback[
    user_article_feedback["exposure_count"] > 1
]

mixed_feedback_pairs = user_article_feedback[
    user_article_feedback["different_labels"] > 1
]

feedback_consistency = pd.Series({
    "Unique exposed user-article pairs": len(user_article_feedback),
    "Pairs exposed more than once": len(repeated_pairs),
    "Pairs observed with both labels": len(mixed_feedback_pairs),
    "Mixed-label percentage among repeated pairs": (
        len(mixed_feedback_pairs) / len(repeated_pairs) * 100
        if len(repeated_pairs) > 0 else 0
    )
})

display(feedback_consistency)

In [ ]:
# Percentages for all article exposures
clicked_pct = (
    feedback_summary.loc[
        feedback_summary["Feedback type"] == "Clicked",
        "Percentage"
    ].iloc[0]
)
unclicked_pct = 100 - clicked_pct

# Percentages among repeatedly exposed user–article pairs
mixed_pct = len(mixed_feedback_pairs) / len(repeated_pairs) * 100
consistent_pct = 100 - mixed_pct

fig, axes = plt.subplots(
    2, 1,
    figsize=(10, 5.5),
    sharex=True
)

bar_height = 0.45

# Panel A: Exposure-level feedback
axes[0].barh(
    0, clicked_pct,
    height=bar_height,
    color="#E76F51",
    edgecolor="white"
)
axes[0].barh(
    0, unclicked_pct,
    left=clicked_pct,
    height=bar_height,
    color="#4C86A8",
    edgecolor="white"
)

# Put the small-segment label above the bar to prevent clipping
axes[0].annotate(
    f"Clicked\n{clicked_pct:.2f}%",
    xy=(clicked_pct / 2, bar_height / 2),
    xytext=(7, 0.48),
    ha="center",
    va="bottom",
    color="#C94F33",
    fontweight="bold",
    arrowprops=dict(
        arrowstyle="-",
        color="#C94F33",
        linewidth=1
    )
)

axes[0].text(
    clicked_pct + unclicked_pct / 2,
    0,
    f"Exposed but unclicked\n{unclicked_pct:.2f}%",
    ha="center",
    va="center",
    color="white",
    fontweight="bold"
)

axes[0].set_title(
    "A. Distribution of Exposure-Level Feedback",
    pad=16,
    fontweight="bold"
)
axes[0].set_yticks([])

# Panel B: Repeated-pair consistency
axes[1].barh(
    0, mixed_pct,
    height=bar_height,
    color="#F4A261",
    edgecolor="white"
)
axes[1].barh(
    0, consistent_pct,
    left=mixed_pct,
    height=bar_height,
    color="#2A9D8F",
    edgecolor="white"
)

axes[1].annotate(
    f"Mixed outcomes\n{mixed_pct:.2f}%",
    xy=(mixed_pct / 2, bar_height / 2),
    xytext=(10, 0.48),
    ha="center",
    va="bottom",
    color="#D97924",
    fontweight="bold",
    arrowprops=dict(
        arrowstyle="-",
        color="#D97924",
        linewidth=1
    )
)

axes[1].text(
    mixed_pct + consistent_pct / 2,
    0,
    f"Consistent outcome\n{consistent_pct:.2f}%",
    ha="center",
    va="center",
    color="white",
    fontweight="bold"
)

axes[1].set_title(
    "B. Feedback Consistency Among Repeated User–Article Pairs",
    pad=16,
    fontweight="bold"
)
axes[1].set_yticks([])
axes[1].set_xlabel("Percentage of observations (%)")

# Shared formatting
for ax in axes:
    ax.set_xlim(0, 100)
    ax.set_ylim(-0.55, 0.85)
    ax.set_xticks(range(0, 101, 20))
    ax.grid(
        axis="x",
        linestyle="--",
        alpha=0.3
    )
    ax.set_axisbelow(True)
    ax.spines[["top", "right", "left"]].set_visible(False)

fig.suptitle(
    "Distribution and Consistency of Implicit Feedback",
    fontsize=14,
    fontweight="bold",
    y=1.02
)

plt.tight_layout(h_pad=2.0)

plt.savefig(
    FIGURE_DIR / "feedback_semantics.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

Repeated user–article pairs can receive both labels across impressions. This shows that a non-click need not indicate a stable dislike: context and competing articles may affect the outcome. The table above reports the counts and mixed-outcome rate for the combined logs.
